# MAPNAI — Agent 2 classifier training (Colab)

Trains MAPNAI's own news classifier for **sports** and **entertainment_movies** (+ `other`).

| Part | What it does | Time on a T4 |
|---|---|---|
| **A. Label** | Qwen2.5-7B-Instruct (open-source, runs here) labels the candidate articles | ~1–1.5 h |
| **B. Train** | Fine-tunes DistilRoBERTa with 4 heads: domain, category, urgency, sentiment | ~10–15 min |
| **C. Evaluate** | Test-set scores, confusion matrix, gold-set export | ~1 min |
| **D. Export** | Zips the model for `models/classifier/` in the MAPNAI repo | ~1 min |

**Before you start**
1. `Runtime → Change runtime type → T4 GPU`.
2. On your PC: `python -m scripts.classifier.make_colab_bundle`, then upload `data/classifier/colab_bundle.zip` to Google Drive at **`MyDrive/MAPNAI_classifier/colab_bundle.zip`**.

Everything is saved to that Drive folder, so if Colab disconnects just re-run **Setup** and continue — labelling resumes where it stopped.

## Setup (run every session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, zipfile, json
WORK = '/content/drive/MyDrive/MAPNAI_classifier'
with zipfile.ZipFile(f'{WORK}/colab_bundle.zip') as z:
    z.extractall(WORK)
sys.path.insert(0, WORK)

!pip -q install -U bitsandbytes accelerate
!nvidia-smi --query-gpu=name,memory.total --format=csv

def load_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

CAND_PATH = f'{WORK}/candidates.jsonl'
LABELED_PATH = f'{WORK}/labeled.jsonl'
REJECTED_PATH = f'{WORK}/rejected.jsonl'
MODEL_DIR = f'{WORK}/model_best'
print(len(load_jsonl(CAND_PATH)), 'candidates |', len(load_jsonl(LABELED_PATH)), 'already labelled')

### Upgrade labels made with an older labelling prompt
If you labelled before the prompt was improved (v2: clearer categories + stricter urgency),
this removes the old labels most likely to be wrong — every movie article and every
article flagged urgent — so the next cell re-labels them. A backup is kept. Safe to re-run.

In [ ]:
import shutil, classifier_taxonomy as tax
rows = load_jsonl(LABELED_PATH)
old = lambda r: r.get('prompt_version', 1) < tax.LABEL_PROMPT_VERSION
redo = [r for r in rows if old(r) and (r['urgency_flag'] or r['domain'] == 'entertainment_movies')]
if redo:
    shutil.copy(LABELED_PATH, LABELED_PATH + f'.backup_v{tax.LABEL_PROMPT_VERSION - 1}')
    keep = [r for r in rows if r not in redo]
    with open(LABELED_PATH, 'w', encoding='utf-8') as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + '\n' for r in keep)
    # Give previously rejected articles another chance with the new prompt too
    if os.path.exists(REJECTED_PATH):
        shutil.move(REJECTED_PATH, REJECTED_PATH + f'.backup_v{tax.LABEL_PROMPT_VERSION - 1}')
print(f'{len(redo)} old labels queued for re-labelling; {len(rows) - len(redo)} kept.')

## Part A — Label the candidates with Qwen2.5-7B-Instruct

Loaded in 4-bit (~5.5 GB VRAM). Output is validated against `classifier_taxonomy.py`; anything
off-taxonomy goes to `rejected.jsonl`. Safe to stop and re-run at any time.

In [ ]:
import re, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import classifier_taxonomy as tax

LABELER = 'Qwen/Qwen2.5-7B-Instruct'
BATCH = 16
MAX_BODY_CHARS = 700

done = {r['id'] for r in load_jsonl(LABELED_PATH)} | {r['id'] for r in load_jsonl(REJECTED_PATH)}
todo = [r for r in load_jsonl(CAND_PATH) if r['id'] not in done]
print(f'{len(done)} done, {len(todo)} to label')

llm_tok = AutoTokenizer.from_pretrained(LABELER, padding_side='left')
llm = AutoModelForCausalLM.from_pretrained(
    LABELER, device_map='auto',
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                           bnb_4bit_compute_dtype=torch.float16),
).eval()
SYSTEM = tax.build_label_prompt()

def to_prompt(r):
    article = json.dumps({'title': r['title'], 'body': r['body'][:MAX_BODY_CHARS]}, ensure_ascii=False)
    return llm_tok.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': article}],
        tokenize=False, add_generation_prompt=True)

def parse(text):
    m = re.search(r'\{.*?\}', text, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

t0, ok, bad = time.time(), 0, 0
with open(LABELED_PATH, 'a', encoding='utf-8') as out, open(REJECTED_PATH, 'a', encoding='utf-8') as rej:
    for s in range(0, len(todo), BATCH):
        batch = todo[s:s + BATCH]
        enc = llm_tok([to_prompt(r) for r in batch], return_tensors='pt', padding=True).to(llm.device)
        with torch.no_grad():
            gen = llm.generate(**enc, max_new_tokens=80, do_sample=False, pad_token_id=llm_tok.pad_token_id)
        texts = llm_tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        for r, t in zip(batch, texts):
            label = tax.validate_label(parse(t))
            if label:
                out.write(json.dumps({**r, **label, 'label_model': LABELER,
                                      'taxonomy_version': tax.TAXONOMY_VERSION,
                                      'prompt_version': tax.LABEL_PROMPT_VERSION}, ensure_ascii=False) + '\n')
                ok += 1
            else:
                rej.write(json.dumps({**r, 'raw_output': t}, ensure_ascii=False) + '\n')
                bad += 1
        out.flush(); rej.flush()
        if (s // BATCH) % 10 == 0:
            rate = (s + len(batch)) / (time.time() - t0)
            print(f'{s + len(batch)}/{len(todo)}  ok={ok} rejected={bad}  |  {rate:.1f} art/s, ~{(len(todo) - s) / rate / 60:.0f} min left')
print(f'Finished: {ok} labelled, {bad} rejected')

In [ ]:
# Free the GPU before training
del llm
import gc; gc.collect(); torch.cuda.empty_cache()

### Label summary — check class balance before training

In [ ]:
import pandas as pd
df = pd.DataFrame(load_jsonl(LABELED_PATH))
print(len(df), 'labelled articles\n')
print(df.domain.value_counts(), '\n')
print(df.groupby(['domain', 'category']).size().to_string(), '\n')
print('urgent share:', round(df.urgency_flag.mean(), 3), '| mean sentiment:', round(df.sentiment.mean(), 3))
print('\nLabeller vs source hint agreement:', round((df.domain == df.hint_domain).mean(), 3))
small = df.category.value_counts()
print('\nCategories with < 100 examples (consider merging):', small[small < 100].to_dict())

## Part B — Train DistilRoBERTa (4 heads)

In [ ]:
import numpy as np, torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
import classifier_taxonomy as tax
import classifier_model as cm

BASE = 'distilroberta-base'
MAX_LEN, EPOCHS, BS, LR = 256, 4, 32, 3e-5
torch.manual_seed(42)
device = 'cuda'

df = pd.DataFrame(load_jsonl(LABELED_PATH))
df['text'] = [cm.NewsClassifierPredictor.format_text(t, b) for t, b in zip(df.title, df.body)]
train_df, rest = train_test_split(df, test_size=0.2, stratify=df.domain, random_state=42)
val_df, test_df = train_test_split(rest, test_size=0.5, stratify=rest.domain, random_state=42)
print(f'train {len(train_df)} | val {len(val_df)} | test {len(test_df)}')

tokenizer = AutoTokenizer.from_pretrained(BASE)
LCFG = tax.label_config(max_length=MAX_LEN)

def encode(d):
    enc = tokenizer(list(d.text), truncation=True, max_length=MAX_LEN)
    return [dict(input_ids=enc['input_ids'][i], domain=tax.DOMAIN_TO_ID[r.domain],
                 category=tax.CATEGORY_TO_ID[r.category], urgency=int(r.urgency_flag),
                 sentiment=float(r.sentiment)) for i, r in enumerate(d.itertuples())]

def collate(items):
    pad = tokenizer.pad({'input_ids': [x['input_ids'] for x in items]}, return_tensors='pt')
    batch = {k: torch.tensor([x[k] for x in items]) for k in ('domain', 'category', 'urgency', 'sentiment')}
    batch.update(input_ids=pad['input_ids'], attention_mask=pad['attention_mask'])
    return batch

train_dl = torch.utils.data.DataLoader(encode(train_df), batch_size=BS, shuffle=True, collate_fn=collate)
val_dl = torch.utils.data.DataLoader(encode(val_df), batch_size=64, collate_fn=collate)

# Urgent articles are rare — up-weight them a little. Kept low (max 3): a big weight makes the
# model call routine news urgent. The decision threshold is tuned after training instead.
n_pos = max(1, int(train_df.urgency_flag.sum()))
URG_POS_WEIGHT = min(3.0, (len(train_df) - n_pos) / n_pos)
print('urgency pos_weight', round(URG_POS_WEIGHT, 2))

# Category weights ~ 1/sqrt(frequency), so big classes (e.g. Box Office) don't swallow small ones
counts = train_df.category.value_counts().reindex(tax.CATEGORIES).fillna(0).clip(lower=1)
cw = 1.0 / np.sqrt(counts.values)
CATEGORY_WEIGHT = torch.tensor(cw / cw.mean(), dtype=torch.float32, device=device)
print('category weights:', dict(zip(tax.CATEGORIES, CATEGORY_WEIGHT.round(decimals=2).tolist())))

In [ ]:
model = cm.build_new(BASE, LCFG).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * len(train_dl)
sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
scaler = torch.cuda.amp.GradScaler()

@torch.no_grad()
def val_metrics():
    model.eval()
    P = {k: [] for k in ('domain', 'category', 'urgency')}; Y = {k: [] for k in P}
    for b in val_dl:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(b['input_ids'], b['attention_mask'])
        P['domain'] += out['domain'].argmax(-1).tolist(); Y['domain'] += b['domain'].tolist()
        P['category'] += out['category'].argmax(-1).tolist(); Y['category'] += b['category'].tolist()
        P['urgency'] += (out['urgency'] > 0).long().tolist(); Y['urgency'] += b['urgency'].tolist()
    model.train()
    return {k: round(f1_score(Y[k], P[k], average='macro'), 4) for k in P}

best = -1
for epoch in range(EPOCHS):
    model.train(); total = 0
    for i, b in enumerate(train_dl):
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(b['input_ids'], b['attention_mask'])
        loss = cm.compute_loss({k: v.float() for k, v in out.items()}, b, URG_POS_WEIGHT, CATEGORY_WEIGHT)
        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        total += loss.item()
    m = val_metrics()
    print(f'epoch {epoch + 1}: train loss {total / len(train_dl):.4f} | val macro-F1 {m}')
    if m['category'] > best:
        best = m['category']
        cm.save_model(model, tokenizer, LCFG, MODEL_DIR)
        print(f'  saved best model (category F1 {best})')

### Tune the urgency threshold on the validation set
Picks the cut-off that maximises F0.5 (precision counts double — a false "urgent" alarm is
worse than a missed one) and saves it into the model's `label_config.json`.

In [ ]:
from sklearn.metrics import fbeta_score, precision_score, recall_score
val_texts = list(val_df.text)
val_probs = np.array(cm.NewsClassifierPredictor(MODEL_DIR).urgency_probabilities(val_texts, batch_size=64))
y = val_df.urgency_flag.values.astype(int)
grid = np.round(np.arange(0.30, 0.991, 0.01), 2)
scores = [fbeta_score(y, val_probs >= t, beta=0.5, zero_division=0) for t in grid]
best_t = float(grid[int(np.argmax(scores))])
p_, r_ = precision_score(y, val_probs >= best_t, zero_division=0), recall_score(y, val_probs >= best_t, zero_division=0)
print(f'best urgency threshold {best_t}  ->  val precision {p_:.3f}, recall {r_:.3f}, F0.5 {max(scores):.3f}')

cfg_path = f'{MODEL_DIR}/label_config.json'
cfg = json.load(open(cfg_path)); cfg['urgency_threshold'] = best_t
json.dump(cfg, open(cfg_path, 'w'), indent=2)

## Part C — Evaluate on the held-out test set (using the exact inference code Agent 2 runs)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, mean_absolute_error

predictor = cm.NewsClassifierPredictor(MODEL_DIR)
preds = pd.DataFrame(predictor.predict(list(test_df.text), batch_size=64), index=test_df.index)

print('=== DOMAIN ===');   print(classification_report(test_df.domain, preds.domain, digits=3))
print('=== CATEGORY ==='); print(classification_report(test_df.category, preds.category, digits=3, zero_division=0))
print('=== URGENCY ===');  print(classification_report(test_df.urgency_flag, preds.urgency_flag, digits=3, zero_division=0))
print('Sentiment MAE:', round(mean_absolute_error(test_df.sentiment, preds.sentiment), 3),
      '| Pearson r:', round(np.corrcoef(test_df.sentiment, preds.sentiment)[0, 1], 3))

fig, ax = plt.subplots(figsize=(11, 9))
ConfusionMatrixDisplay(confusion_matrix(test_df.category, preds.category, labels=tax.CATEGORIES),
                       display_labels=tax.CATEGORIES).plot(ax=ax, xticks_rotation=60, colorbar=False)
ax.set_title('Category confusion matrix (test set)'); plt.tight_layout(); plt.show()

mistakes = test_df.assign(pred_domain=preds.domain, pred_category=preds.category)
mistakes = mistakes[mistakes.category != mistakes.pred_category]
mistakes[['title', 'body', 'category', 'pred_category']].head(15)

### Gold set (recommended for your report)
The scores above compare against Qwen's labels. For a trustworthy number, export 300 test
articles, correct the labels by hand in Google Sheets (`gold_*` columns), save the result as
`gold_reviewed.csv` in the same Drive folder, then run the next cell again.

In [ ]:
GOLD_OUT, GOLD_IN = f'{WORK}/gold_to_review.csv', f'{WORK}/gold_reviewed.csv'
if not os.path.exists(GOLD_OUT):
    g = test_df.sample(n=min(300, len(test_df)), random_state=7)
    g.assign(gold_domain=g.domain, gold_category=g.category, gold_urgency=g.urgency_flag)[
        ['id', 'title', 'body', 'gold_domain', 'gold_category', 'gold_urgency']].to_csv(GOLD_OUT, index=False)
    print('Wrote', GOLD_OUT, '— review it, save as gold_reviewed.csv, and re-run this cell.')
if os.path.exists(GOLD_IN):
    gold = pd.read_csv(GOLD_IN)
    gp = pd.DataFrame(predictor.predict([cm.NewsClassifierPredictor.format_text(t, b)
                                         for t, b in zip(gold.title, gold.body.fillna(''))]))
    print('GOLD domain accuracy  :', round((gp.domain == gold.gold_domain).mean(), 3))
    print('GOLD category accuracy:', round((gp.category == gold.gold_category).mean(), 3))
    print('GOLD urgency accuracy :', round((gp.urgency_flag == gold.gold_urgency.astype(bool)).mean(), 3))

## Part D — Export the model for MAPNAI

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive(f'{WORK}/mapnai_classifier_model', 'zip', MODEL_DIR)
print('Saved', zip_path)
files.download(zip_path)

**On your PC:** unzip `mapnai_classifier_model.zip` into `models/classifier/` in the MAPNAI repo
(so that `models/classifier/label_config.json` exists). Then test it:

```
python -m agents.agent2_classifier --text "Kohli ruled out of the World Cup final with a hamstring injury"
```